In [106]:
import torch
import torch.nn.functional as F
from PIL import Image
from pathlib import Path
import json
from torchvision.models import resnet34, ResNet34_Weights
from torchvision import transforms
from ImageNet100ValDataset import ImageNet100ValDataset, transform
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import save_image


# -------------------------
# Configuración base
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = ResNet34_Weights.DEFAULT
model = resnet34(weights=weights).to(device).eval()
imagenet_classes = weights.meta["categories"]

vis_transform = transform

# -------------------------
# Cargar mapping WNID <-> nombre
# -------------------------
with open("Labels.json", "r") as f:
    wnid2name = json.load(f)

# Carpetas del dataset (100 clases)
dataset_root = Path("val.X")
selected_classes = sorted([p.name for p in dataset_root.iterdir() if p.is_dir()])

# -------------------------
# Mapeo robusto WNID → índice de clase del modelo
# -------------------------
wnid_to_model_idx = {}
for wnid in selected_classes:
    if wnid not in wnid2name:
        continue
    wnid_name = wnid2name[wnid].split(",")[0].lower().strip()
    # Buscar coincidencia dentro de las 1000 clases de ImageNet
    match = [i for i, c in enumerate(imagenet_classes) if wnid_name in c.lower()]
    if match:
        wnid_to_model_idx[wnid] = match[0]

selected_indices_in_model = list(wnid_to_model_idx.values())
model_idx_to_wnid = {v: k for k, v in wnid_to_model_idx.items()}

print(f"Total carpetas detectadas en val.X: {len(selected_classes)}")
print(f"Total clases mapeadas correctamente: {len(selected_indices_in_model)}")

if not selected_indices_in_model:
    raise ValueError("No se encontró ninguna clase de val.X en las 1000 del modelo. Verifica Labels.json y los nombres de carpeta.")

# -------------------------
# Función Top-5 (solo tus 100 clases)
# -------------------------
def top5_for_image_path(model, img_path, preprocess, device=device):
    img = Image.open(img_path).convert("RGB")
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)[0]
        logits_filtered = logits[selected_indices_in_model]
        probs = F.softmax(logits_filtered, dim=0)

    k = min(5, len(selected_indices_in_model))
    topk = torch.topk(probs, k=k)
    results = []
    for idx_f, p in zip(topk.indices.cpu().tolist(), topk.values.cpu().tolist()):
        model_idx = selected_indices_in_model[idx_f]
        wnid = model_idx_to_wnid[model_idx]
        wnid_name = wnid2name.get(wnid, "desconocido")
        class_name = imagenet_classes[model_idx]
        results.append({
            "model_idx": model_idx,
            "class_name": class_name,
            "prob": p,
            "wnid": wnid,
            "wnid_name": wnid_name
        })
    return results


def mostrar_diferencia(img_path_original, img_path_adv, out_path="diff_visual.png"):
    """
    Muestra y guarda una visualización de la diferencia entre dos imágenes (original y adversarial).
    
    Args:
        img_path_original (str): Ruta a la imagen original.
        img_path_adv (str): Ruta a la imagen adversarial.
        out_path (str): Ruta de salida para guardar la imagen comparativa.
    """
    # Cargar imágenes
    img_orig = Image.open(img_path_original).convert("RGB")
    img_adv = Image.open(img_path_adv).convert("RGB")

    # Convertir a tensores normalizados [0, 1]

    t_orig = vis_transform(img_orig)
    t_adv = vis_transform(img_adv)

    # Calcular diferencia absoluta
    diff = torch.abs(t_adv - t_orig)

    # Escalar diferencia para visualizar mejor (opcional)
    diff_vis = diff / diff.max()

    # Convertir a numpy para graficar
    orig_np = np.transpose(t_orig.numpy(), (1, 2, 0))
    adv_np = np.transpose(t_adv.numpy(), (1, 2, 0))
    diff_np = np.transpose(diff_vis.numpy(), (1, 2, 0))

    # Crear figura
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig_np)
    axes[0].set_title("Original")
    axes[1].imshow(adv_np)
    axes[1].set_title("Adversarial")
    axes[2].imshow(diff_np)
    axes[2].set_title("Diferencia (resaltada)")
    
    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight", dpi=300)
    plt.close(fig)

    print(f"Imagen de diferencia guardada en: {Path(out_path).resolve()}")

def guardar_diferencia(img_path_original, img_path_adv, out_path="diff_only.png", amplify=5.0):
    """
    Guarda solo la imagen de la diferencia entre dos imágenes (original y adversarial).
    
    Args:
        img_path_original (str): Ruta a la imagen original.
        img_path_adv (str): Ruta a la imagen adversarial.
        out_path (str): Ruta de salida para guardar la imagen de diferencia.
        amplify (float): Factor para amplificar las diferencias pequeñas (opcional).
    """
    # Cargar imágenes
    img_orig = Image.open(img_path_original).convert("RGB")
    img_adv = Image.open(img_path_adv).convert("RGB")

    # Convertir a tensores normalizados [0, 1]
    t_orig = vis_transform(img_orig)
    t_adv = vis_transform(img_adv)

    # Calcular diferencia absoluta
    diff = torch.abs(t_adv - t_orig)

    # Amplificar para hacer visibles diferencias pequeñas (opcional)
    diff = torch.clamp(diff * amplify, 0, 1)

    # Convertir a imagen PIL
    to_pil = transforms.ToPILImage()
    diff_img = to_pil(diff)

    # Guardar imagen
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    diff_img.save(out_path)

    print(f"Imagen de diferencia guardada en: {out_path.resolve()}")

def guardar_png_transformado(img_path, out_path, preprocess=None):
    """
    Convierte una imagen (por ejemplo JPEG) aplicando la transformación geométrica
    usada por los ataques (Resize + CenterCrop + ToTensor) y guarda un PNG.

    Args:
        img_path (str or Path): Ruta de la imagen original (JPEG, etc.)
        out_path (str or Path): Ruta donde guardar el PNG resultante.
        preprocess (torchvision.transforms.Compose, opcional): Transformación del modelo
            (por ejemplo, weights.transforms()). Si incluye Normalize, se elimina automáticamente.

    Returns:
        Path: ruta del archivo PNG generado.
    """
    img_path = Path(img_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Si se pasa un preprocess del modelo (p.ej. ResNet34_Weights.DEFAULT.transforms()),
    # eliminamos la normalización final (solo queremos la parte geométrica).
    def remove_normalize(transform):
        if hasattr(transform, "transforms") and isinstance(transform.transforms, list):
            ts = [t for t in transform.transforms if not isinstance(t, transforms.Normalize)]
            return transforms.Compose(ts)
        return transform

    if preprocess is not None:
        transform = remove_normalize(preprocess)
    else:
        # Transformación por defecto (como ImageNet)
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor()
        ])

    # Cargar imagen y aplicar transformaciones
    img = Image.open(img_path).convert("RGB")
    tensor = transform(img)  # tensor [C,H,W] en [0,1]

    # Guardar el PNG resultante
    save_image(tensor, str(out_path))

    return out_path


Total carpetas detectadas en val.X: 100
Total clases mapeadas correctamente: 100


In [107]:
img_path = r"val.X\n01491361\ILSVRC2012_val_00029300.JPEG"
img_path_noisy = r"val_noisy\n01491361\ILSVRC2012_val_00029300_noisy.jpeg"
img_path_FGSM = r"FGSM_out\n01491361\ILSVRC2012_val_00029300_fgsm.png" 
img_path_RFGSM = r"RFGSM_out\n01491361\img_000476_rfgsm.png"
img_path_PGD = r"PGD_untargeted_out\n01491361\img_000451_pgd_untargeted.png"
  
res = top5_for_image_path(model, img_path, transform, device=device)
print("Top-5 imagen pura:")
for r in res:
    print(f"{r['model_idx']:4d}  {r['class_name'][:35]:35s}  p={r['prob']:.4f}  wnid={r['wnid']}  wnid_name={r['wnid_name']}")

   
res = top5_for_image_path(model, img_path_noisy, transform, device=device)
print("\nTop-5 imagen por ruido blanco:")
for r in res:
    print(f"{r['model_idx']:4d}  {r['class_name'][:35]:35s}  p={r['prob']:.4f}  wnid={r['wnid']}  wnid_name={r['wnid_name']}")

res = top5_for_image_path(model, img_path_FGSM, transform, device=device)
print("\nTop-5 imagen por FGSM:")
for r in res:
    print(f"{r['model_idx']:4d}  {r['class_name'][:35]:35s}  p={r['prob']:.4f}  wnid={r['wnid']}  wnid_name={r['wnid_name']}")

res = top5_for_image_path(model, img_path_RFGSM, transform, device=device)
print("\nTop-5 imagen por RFGSM:")
for r in res:
    print(f"{r['model_idx']:4d}  {r['class_name'][:35]:35s}  p={r['prob']:.4f}  wnid={r['wnid']}  wnid_name={r['wnid_name']}")

  
res = top5_for_image_path(model, img_path_PGD, transform, device=device)
print("\nTop-5 imagen por PDG:")
for r in res:
    print(f"{r['model_idx']:4d}  {r['class_name'][:35]:35s}  p={r['prob']:.4f}  wnid={r['wnid']}  wnid_name={r['wnid_name']}")

Top-5 imagen pura:
   3  tiger shark                          p=0.9929  wnid=n01491361  wnid_name=tiger shark, Galeocerdo cuvieri
   2  great white shark                    p=0.0044  wnid=n01484850  wnid_name=great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias
   6  stingray                             p=0.0017  wnid=n01498041  wnid_name=stingray
   5  electric ray                         p=0.0006  wnid=n01496331  wnid_name=electric ray, crampfish, numbfish, torpedo
   4  hammerhead                           p=0.0004  wnid=n01494475  wnid_name=hammerhead, hammerhead shark

Top-5 imagen por ruido blanco:
   3  tiger shark                          p=0.9894  wnid=n01491361  wnid_name=tiger shark, Galeocerdo cuvieri
   2  great white shark                    p=0.0052  wnid=n01484850  wnid_name=great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias
   6  stingray                             p=0.0029  wnid=n01498041  wnid_name=stin

In [108]:
weights = ResNet34_Weights.DEFAULT
preprocess = weights.transforms()

ruta_png = guardar_png_transformado(img_path, "imagen_original.png", preprocess=transform)
print(f"Imagen convertida guardada en: {ruta_png}")
img_path_noisy = guardar_png_transformado(img_path_noisy, "imagen_noisy.png", preprocess=transform)

guardar_diferencia(ruta_png, img_path_noisy, out_path="diff_original_noisy.png", amplify=1)
guardar_diferencia(ruta_png, img_path_FGSM, out_path="diff_original_FGSM.png", amplify=1)
guardar_diferencia(ruta_png, img_path_RFGSM, out_path="diff_original_RFGSM.png", amplify=1)
guardar_diferencia(ruta_png, img_path_PGD, out_path="diff_original_PGD.png", amplify=1)

Imagen convertida guardada en: imagen_original.png
Imagen de diferencia guardada en: C:\Users\Benjamin\OneDrive - Universidad de Chile\Escritorio\Tareas progra\Inteligencia compuacional\Defensa-adversaria-en-redes-neuronales-con-ImagenNet\diff_original_noisy.png
Imagen de diferencia guardada en: C:\Users\Benjamin\OneDrive - Universidad de Chile\Escritorio\Tareas progra\Inteligencia compuacional\Defensa-adversaria-en-redes-neuronales-con-ImagenNet\diff_original_FGSM.png
Imagen de diferencia guardada en: C:\Users\Benjamin\OneDrive - Universidad de Chile\Escritorio\Tareas progra\Inteligencia compuacional\Defensa-adversaria-en-redes-neuronales-con-ImagenNet\diff_original_RFGSM.png
Imagen de diferencia guardada en: C:\Users\Benjamin\OneDrive - Universidad de Chile\Escritorio\Tareas progra\Inteligencia compuacional\Defensa-adversaria-en-redes-neuronales-con-ImagenNet\diff_original_PGD.png


In [109]:
img_path_original = r"val.X\n01440764\ILSVRC2012_val_00000293.JPEG"
img_path_FGSM = r"FGSM_targeted_n01883070/n01440764/ILSVRC2012_val_00000293_fgsm_targeted.png"
img_path_RFGSM = r"RFGSM_dirigido_target_106/n01440764/img_000650_rfgsm_target_106.png"
img_path_PGD = r"PGD_dirigido_target_106_zip (1)/n01440764/img_000650_pgd_target_106.png"

res = top5_for_image_path(model, img_path_original, transform, device=device)
print("Top-5 Imagen Original:")
for r in res:
    print(f"{r['model_idx']:4d}  {r['class_name'][:35]:35s}  p={r['prob']:.4f}  wnid={r['wnid']}  wnid_name={r['wnid_name']}")

   
res = top5_for_image_path(model, img_path_FGSM, transform, device=device)
print("\nTop-5 Imagen para wombat con FGSM:")
for r in res:
    print(f"{r['model_idx']:4d}  {r['class_name'][:35]:35s}  p={r['prob']:.4f}  wnid={r['wnid']}  wnid_name={r['wnid_name']}")

   
res = top5_for_image_path(model, img_path_RFGSM, transform, device=device)
print("\nTop-5 Imagen para wombat con FGSM:")
for r in res:
    print(f"{r['model_idx']:4d}  {r['class_name'][:35]:35s}  p={r['prob']:.4f}  wnid={r['wnid']}  wnid_name={r['wnid_name']}")

res = top5_for_image_path(model, img_path_PGD, transform, device=device)
print("\nTop-5 Imagen para wombat con PGD:")
for r in res:
    print(f"{r['model_idx']:4d}  {r['class_name'][:35]:35s}  p={r['prob']:.4f}  wnid={r['wnid']}  wnid_name={r['wnid_name']}")


Top-5 Imagen Original:
   0  tench                                p=0.9802  wnid=n01440764  wnid_name=tench, Tinca tinca
  48  Komodo dragon                        p=0.0031  wnid=n01695060  wnid_name=Komodo dragon, Komodo lizard, dragon lizard, giant lizard, Varanus komodoensis
   1  goldfish                             p=0.0027  wnid=n01443537  wnid_name=goldfish, Carassius auratus
   3  tiger shark                          p=0.0026  wnid=n01491361  wnid_name=tiger shark, Galeocerdo cuvieri
  29  axolotl                              p=0.0021  wnid=n01632777  wnid_name=axolotl, mud puppy, Ambystoma mexicanum

Top-5 Imagen para wombat con FGSM:
  33  loggerhead                           p=0.2632  wnid=n01664065  wnid_name=loggerhead, loggerhead turtle, Caretta caretta
  48  Komodo dragon                        p=0.2079  wnid=n01695060  wnid_name=Komodo dragon, Komodo lizard, dragon lizard, giant lizard, Varanus komodoensis
 106  wombat                               p=0.1203  wnid=n01883

In [110]:
# compare_perturbations.py
import torch
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from PIL import Image
from pathlib import Path
import os

# -------------------------
# Preprocess que asegura mismo recorte/resize (sin Normalize)
# -------------------------
DEFAULT_PREPROCESS = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),   # resultado en [0,1]
])

# -------------------------
# Función principal
# -------------------------
def make_perturbation_comparison(
    img_path_original,
    adv_paths_dict,
    out_path="comparison.png",
    preprocess=None,
    amplify=1.0,
    show_adv_also=False
):
    """
    Crea y guarda una imagen con: [Original | diff_method1 | diff_method2 | ...]
    - img_path_original: ruta a JPEG original
    - adv_paths_dict: dict {"FGSM": "ruta/fgsm.png", "PGD": "ruta/pgd.png", ...}
    - out_path: ruta donde se guardará la imagen resultante (PNG)
    - preprocess: torchvision transform para homogeneizar tamaño
    - amplify: factor multiplicativo sobre la diferencia
    - show_adv_also: si True incluye imagen adversarial junto a cada diff
    """
    preprocess = preprocess or DEFAULT_PREPROCESS
    img_path_original = Path(img_path_original)

    pil_orig = Image.open(img_path_original).convert("RGB")
    t_orig = preprocess(pil_orig)  # [C,H,W], [0,1]

    cols = 1 + len(adv_paths_dict) * (2 if show_adv_also else 1)
    tensors_for_grid = []
    labels = []

    tensors_for_grid.append(t_orig)
    labels.append("Original")

    for method_name, adv_path in adv_paths_dict.items():
        adv_path = Path(adv_path)
        if not adv_path.exists():
            raise FileNotFoundError(f"Adv image for '{method_name}' not found at {adv_path}")

        pil_adv = Image.open(adv_path).convert("RGB")
        t_adv = preprocess(pil_adv)

        # diferencia real (sin normalizar)
        diff = (t_adv - t_orig) * float(amplify)

        # clip a [0,1] para guardarla sin artefactos
        diff_vis = torch.clamp(diff, 0, 1)

        tensors_for_grid.append(diff_vis)
        labels.append(f"Perturbación: {method_name}")

        if show_adv_also:
            tensors_for_grid.append(t_adv)
            labels.append(f"Adversaria: {method_name}")

    grid = make_grid(tensors_for_grid, nrow=len(tensors_for_grid), padding=8, pad_value=1.0)

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    save_image(grid, str(out_path))

    txt_labels = out_path.with_suffix(".labels.txt")
    with open(txt_labels, "w") as f:
        for i, lab in enumerate(labels):
            f.write(f"{i}: {lab}\n")

    print(f"Comparativa guardada en: {out_path.resolve()}")
    print(f"Archivo con etiquetas guardado en: {txt_labels.resolve()}")
    return out_path

# -------------------------
# Ejemplo de uso
# -------------------------
if __name__ == "__main__":
    # Rutas de ejemplo (ajusta a tus paths reales)
    img_path = r"val.X\n01440764\ILSVRC2012_val_00000293.JPEG"
    ruta_png = guardar_png_transformado(img_path, "imagen_original.png", preprocess=transform)

    img_path_noisy = r"val_noisy\n01440764\ILSVRC2012_val_00000293_noisy.jpeg"
    img_path_noisy = guardar_png_transformado(img_path_noisy, "imagen_noisy.png", preprocess=transform)

    img_path_FGSM = r"FGSM_out\n01440764\ILSVRC2012_val_00000293_fgsm.png" 
    img_path_RFGSM = r"RFGSM_out\n01440764\img_00050_rfgsm.png"
    img_path_PGD = r"PGD_untargeted_out\n01440764\img_000684_pgd_untargeted.png"

    advs = {
        "Gaussian": img_path_noisy,
        "FGSM": img_path_FGSM,
        "R-FGSM": img_path_RFGSM,
        "PGD": img_path_PGD
    }

    make_perturbation_comparison(
        ruta_png,
        advs,
        out_path="comparativa_original_vs_diffs.png",
        preprocess=DEFAULT_PREPROCESS,
        amplify=1.0,
        show_adv_also=False
    )


Comparativa guardada en: C:\Users\Benjamin\OneDrive - Universidad de Chile\Escritorio\Tareas progra\Inteligencia compuacional\Defensa-adversaria-en-redes-neuronales-con-ImagenNet\comparativa_original_vs_diffs.png
Archivo con etiquetas guardado en: C:\Users\Benjamin\OneDrive - Universidad de Chile\Escritorio\Tareas progra\Inteligencia compuacional\Defensa-adversaria-en-redes-neuronales-con-ImagenNet\comparativa_original_vs_diffs.labels.txt
